<a href="https://colab.research.google.com/github/ha723-web/AI-Based-Youtube-Video-Summarizer-/blob/main/Another_copy_of_BERT_Fine_Tuning_for_Customer_Review_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **Step 0 — Install libraries in Colab**

In [ ]:
!pip install -q transformers datasets accelerate


## **Step 1 — Import Libraries**

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset, DatasetDict
import torch
import pandas as pd
import numpy as np
import os

os.environ["WANDB_DISABLED"] = "true"


## **Step 2 — Sample dataset**

In [ ]:
# Sample customer review data
reviews_data = {
    'text': [
        "This product exceeded my expectations. Great quality and fast shipping!",
        "Terrible experience. Product broke after one day of use.",
        "Average product, nothing special but does the job.",
        "Outstanding customer service and excellent product quality.",
        "Would not recommend. Poor build quality and overpriced."
    ],
    'label': [1, 0, 2, 1, 0]   # 0: negative, 1: positive, 2: neutral
}

df = pd.DataFrame(reviews_data)

print("Dataset shape:", df.shape)
print("\nSample data:")
print(df.head())


Dataset shape: (5, 2)

Sample data:
                                                text  label
0  This product exceeded my expectations. Great q...      1
1  Terrible experience. Product broke after one d...      0
2  Average product, nothing special but does the ...      2
3  Outstanding customer service and excellent pro...      1
4  Would not recommend. Poor build quality and ov...      0


## **Step 3 — Initialize tokenizer & model**

In [ ]:
# Initialize tokenizer and model
model_name = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3   # positive, negative, neutral
)

print(f"Loaded model: {model_name}")
print(f"Model parameters: {model.num_parameters():,}")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Loaded model: bert-base-uncased
Model parameters: 109,484,547


## **Step 4 — Tokenization**

In [ ]:
def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        truncation=True,
        padding=True,
        max_length=128
    )

# Convert to Hugging Face dataset format
dataset = Dataset.from_pandas(df)
tokenized_dataset = dataset.map(tokenize_function, batched=True)

print("Tokenization complete")
print("Dataset features:", tokenized_dataset.features)


Map:   0%|          | 0/5 [00:00<?, ? examples/s]

Tokenization complete
Dataset features: {'text': Value('string'), 'label': Value('int64'), 'input_ids': List(Value('int32')), 'token_type_ids': List(Value('int8')), 'attention_mask': List(Value('int8'))}


## **Step 5 — TrainingArguments**

In [ ]:
!pip install -q --upgrade transformers datasets accelerate


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.6/511.6 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 14.0 MB/s eta 0:00:00


In [ ]:
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    warmup_steps=10,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=1,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none",          # NEW: tell Trainer not to use wandb
)


TypeError: TrainingArguments.__init__() got an unexpected keyword argument 'evaluation_strategy'

## **Step 6 — Trainer initialization**

In [ ]:
# Split dataset for training and evaluation
train_size = int(0.8 * len(tokenized_dataset))
train_dataset = tokenized_dataset.select(range(train_size))
eval_dataset = tokenized_dataset.select(range(train_size, len(tokenized_dataset)))

# Initialize trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
)

print("Trainer initialized successfully")
print(f"Training samples: {len(train_dataset)}")
print(f"Evaluation samples: {len(eval_dataset)}")


NameError: name 'training_args' is not defined

## **Step 7 — Training**

In [ ]:
# Start training
print("Starting fine-tuning...")
trainer.train()
print("\nTraining completed!")


Starting fine-tuning...


wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

## **Step 8 — Testing the fine-tuned model**

In [ ]:
# Test the fine-tuned model
test_text = "This product is amazing and works perfectly!"

inputs = tokenizer(test_text, return_tensors="pt", truncation=True, padding=True)

with torch.no_grad():
    outputs = model(**inputs)  # note the ** to unpack the dict
    predictions = torch.nn.functional.softmax(outputs.logits, dim=-1)
    predicted_class = torch.argmax(predictions, dim=-1)

labels = ["negative", "positive", "neutral"]

print(f"Text: {test_text}")
print(f"Predicted class: {labels[predicted_class.item()]}")
print(f"Confidence scores: {predictions.detach().cpu().numpy()[0]}")


Text: This product is amazing and works perfectly!
Predicted class: positive
Confidence scores: [0.30386245 0.3879217  0.30821586]
